# QLoRA v3E — Clean-label 9B ablation on Google Colab

**Goal:** Repeat the published v3B Qwen3.5-9B all-linear 4-bit LoRA experiment using exactly 64 optimizer steps while changing only the supervised action selection to **v3e-cleaned**. The old v3B labels included 4 deliberately failed first tool calls. No GAIA tasks are used for training.

**Status:** This notebook is an executable recipe, **not a claim that Colab GPU training has occurred or that GAIA accuracy improved**. Use **Runtime → Change runtime type → GPU**, preferably L4/A100; T4 may run out of memory or be too slow. Colab runtime, GPU selection, Google account authorization and quota require your interaction.

- Source pin: `204dab90bd37692d59cdde344c3b9ff594d5f4e2` on `research/verified-teacher-trajectory-v0-20261112`.
- Frozen settings: Qwen/Qwen3.5-9B, NF4 4-bit, all-linear LoRA rank 16 / alpha 32 / dropout 0.05, LR 1e-4, 64 steps, length 1024, grad accumulation 1.
- First perform a CPU/tokenization preflight; **abort before GPU optimization** if any invariants fail.
- Do not substitute the small HotpotQA pilot (only 2 verified long trajectories) as training data.
- The adapter and audit metadata remain local until you explicitly save/download them. No secrets are printed.

Colab GPUs are allocated by Google, not by this notebook.

In [ ]:
import os, sys, subprocess, platform
print("Python:", sys.version.split()[0], "| OS:", platform.platform())
print("GPU status:")
subprocess.run(["nvidia-smi"], check=True)
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > GPU"
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / (1024 ** 3)
print(f"GPU: {gpu.name}, memory: {vram_gib:.1f} GiB, CUDA: {torch.version.cuda}")
if vram_gib < 20:
    print("CAUTION: 9B all-linear 4-bit QLoRA may OOM or be slow on this GPU. Do not silently switch to 4B.")


In [ ]:
# Install the exact public research-branch implementation and its training dependencies.
# To reproduce, NEVER train from a moving branch: detach to the pinned SHA.
import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/small-agent-qlora")
BRANCH = "research/verified-teacher-trajectory-v0-20261112"
SOURCE_SHA = "204dab90bd37692d59cdde344c3b9ff594d5f4e2"
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--branch", BRANCH,
         "https://github.com/kolihhan/small-agent-qlora.git", str(REPO)],
        check=True,
    )
else:
    assert (REPO / ".git").is_dir(), "Existing /content/small-agent-qlora is not a Git clone"
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "100", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_SHA], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == SOURCE_SHA, (actual, SOURCE_SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[train,files,dev]", "jinja2>=3.1"], check=True)
print("Pinned repository revision:", actual)


In [ ]:
# Regenerate the SAME deterministic, non-GAIA oracle training corpus used for v3B.
import json, os, tempfile
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone

from gaia_small_agent.training.oracle_policy_v3 import generate_oracle_policy_v3_dataset
from gaia_small_agent.training.qlora import load_verified_rows

RUN_ID = "colab-v3e-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = Path("/content") / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
DATA = OUT / "oracle"
paths = generate_oracle_policy_v3_dataset(DATA, seed="small-agent-policy-v3")
rows = load_verified_rows(paths["train"])
assert len(rows) == 1600, f"Unexpected frozen source count: {len(rows)}"
print("Frozen synthetic training tasks:", len(rows))
print("Output directory:", OUT)


In [ ]:
# CPU-friendly preflight. Download tokenizer only; no 9B model weights yet.
from transformers import AutoTokenizer
from gaia_small_agent.training.qlora import (
    _render_turn_examples,
    _select_diagnostic_exposure_examples,
    _select_v3e_clean_exposure_examples,
    _tokenize_completion_examples,
    _build_training_exposure_report,
)

MODEL = "Qwen/Qwen3.5-9B"
TOKENIZER = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
all_examples = _render_turn_examples(TOKENIZER, rows)
old = _select_diagnostic_exposure_examples(all_examples, count=64, seed="qlora-v3b-exposure")
clean = _select_v3e_clean_exposure_examples(all_examples, count=64, seed="qlora-v3b-exposure")

def deliberately_failed_first_call(example):
    meta = example["metadata"]
    return (
        meta["capability"] in {"failure_recovery", "strategy_switch"}
        and meta["action_type"] == "tool_call"
        and meta["assistant_message_index"] == 2
    )

assert len(all_examples) == 3600
assert len(old) == len(clean) == 64
assert sum(deliberately_failed_first_call(x) for x in old) == 4
assert sum(deliberately_failed_first_call(x) for x in clean) == 0
assert len({x["metadata"]["task_id"] for x in clean}) == 64
assert Counter(x["metadata"]["action_type"] for x in clean) == Counter({"tool_call": 35, "final": 29})
assert {x["metadata"]["example_id"] for x in old if x["metadata"]["capability"] not in {"failure_recovery","strategy_switch"}} == {
    x["metadata"]["example_id"] for x in clean if x["metadata"]["capability"] not in {"failure_recovery","strategy_switch"}
}
tokenized = _tokenize_completion_examples(
    TOKENIZER, clean, max_length=1024, require_full_prompt=True
)
assert len(tokenized) == 64
assert all(any(label != -100 for label in ex["labels"]) for ex in tokenized)
report = _build_training_exposure_report(clean, seed="qlora-v3b-exposure", source_example_count=len(all_examples))
report.update({"profile":"v3e-cleaned", "known_bad_labels_v3b":4, "known_bad_labels_v3e":0, "tokenized_examples":len(tokenized)})
assert all(len(x["input_ids"]) <= 1024 for x in tokenized)
(OUT / "preflight.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Preflight PASS:", len(all_examples), "source actions;", len(clean), "train labels;", "known-bad labels 4→0")
print("Max token length:", max(len(x["input_ids"]) for x in tokenized))
print("Exposure selection SHA256:", report["selection_sha256"])


In [ ]:
# Train only if the exact 64-label/token preflight succeeded.
# Uses the repository's existing, tested training implementation.
import torch
from gaia_small_agent.training.qlora import train_qlora
assert torch.cuda.is_available(), "No NVIDIA GPU connected"
assert len(clean) == 64 and report["known_bad_labels_v3e"] == 0
ADAPTER = OUT / "adapter"
assert not ADAPTER.exists(), "Refusing to overwrite an existing adapter"
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
train_qlora(
    data_path=paths["train"],
    output_dir=ADAPTER,
    model_name=MODEL,
    max_length=1024,
    learning_rate=1e-4,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    max_steps=64,
    save_steps=64,
    gradient_accumulation_steps=1,
    lora_target_modules="all-linear",
    exposure_count=64,
    exposure_seed="qlora-v3b-exposure",
    exposure_profile="v3e-cleaned",
    require_full_prompt=True,
)
print("Finished 64 optimizer steps, pending output validation:", ADAPTER)


In [ ]:
# Verify *real* adapter bytes before any success claim; save separately from the code repo.
import json, hashlib, shutil
from pathlib import Path

for required in ["adapter_config.json", "adapter_model.safetensors", "run_config.json", "training-exposure.json"]:
    assert (ADAPTER / required).is_file(), f"Missing output: {required}"
cfg = json.loads((ADAPTER / "run_config.json").read_text(encoding="utf-8"))
exposure = json.loads((ADAPTER / "training-exposure.json").read_text(encoding="utf-8"))
assert cfg["max_steps"] == 64 and cfg["max_length"] == 1024
assert cfg["require_full_prompt"] is True
assert cfg["lora_target_modules"] == "all-linear"
assert cfg["exposure_profile"] == "v3e-cleaned"
assert exposure["selection_sha256"] == report["selection_sha256"]
assert exposure["selected_example_count"] == 64
assert not any(deliberately_failed_first_call({"metadata": item}) for item in exposure["examples"])
weights = ADAPTER / "adapter_model.safetensors"
assert weights.stat().st_size > 100_000, "Adapter weights file appears empty"
archive = shutil.make_archive(str(OUT / "qwen35-9b-qlora-v3e-adapter"), "zip", ADAPTER)
sha = hashlib.file_digest(open(archive, "rb"), "sha256").hexdigest()
(OUT / "adapter_sha256.txt").write_text(f"{sha}  {Path(archive).name}\n", encoding="utf-8")
print("VERIFIED: 64-step v3E adapter saved at", archive)
print("SHA256:", sha)
print("Bytes:", Path(archive).stat().st_size)
print("Do not claim a GAIA accuracy improvement until a separate paired evaluation.")


In [ ]:
# Explicit Google Drive persistence step. Colab will request authorization.
# Output folder is unique; old v3B/v3D adapters are never overwritten.
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/qlora-v3e-colab") / RUN_ID
destination.mkdir(parents=True, exist_ok=False)
for item in [Path(archive), OUT / "preflight.json", OUT / "adapter_sha256.txt"]:
    shutil.copy2(item, destination / item.name)
print("Saved adapter, SHA256 and audit to:", destination)
print("Next: share the adapter zip with the evaluation workflow; do not train on GAIA test cases.")


## After training — measure, don't assume

The notebook saves a genuine PEFT adapter and its hash. It **does not** run GAIA. The following evaluation must compare **v3B vs v3E** using the same GGUF conversion/runtime, frozen synthetic heldout 64, frozen 200 task control and untouched external GAIA split, while measuring completion and duplicate calls. Do not merge v3E simply because the loss decreases.

If Colab disconnects before the adapter is archived, no training success can be claimed. Do not switch silently to a smaller model, a different max length or a different LoRA target list: that would cease to be a controlled v3B-v3E comparison.